# Sequential Feature Selection: SFS, SBS, and SFFS from Scratch

## Learning Objectives

By completing this notebook, you will:
1. Implement Sequential Forward Selection (SFS), Sequential Backward Selection (SBS), and Sequential Floating Forward Selection (SFFS) from scratch
2. Apply all three methods to a real dataset and compare the selected feature sets
3. Visualise the search path — which features were added or removed at each step
4. Measure and compare the computational cost of each method
5. Understand when the floating mechanism changes the result versus basic greedy search

## Prerequisites
- Module 01: Statistical filter methods
- Module 02: Information-theoretic feature selection
- Cross-validation with scikit-learn

## Estimated Time: 15 minutes

---

## 1. Setup

We use the **OpenML diabetes dataset** (Pima Indians Diabetes) — 768 samples, 8 features — small enough to run all methods in a few seconds while demonstrating the key differences between SFS, SBS, and SFFS.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.pipeline import Pipeline

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print('Libraries loaded.')

In [ ]:
# Load Pima Indians Diabetes dataset from OpenML
# 768 samples, 8 features: pregnancies, glucose, blood pressure, skin thickness,
# insulin, BMI, diabetes pedigree function, age → binary outcome
diabetes = fetch_openml('diabetes', version=1, as_frame=True, parser='liac-arff')
X_df = diabetes.data
y_raw = diabetes.target

# Encode target: tested_positive=1, tested_negative=0
y = (y_raw == 'tested_positive').astype(int).values
X = X_df.values.astype(float)
feature_names = list(X_df.columns)

print(f'Dataset shape: {X.shape}')
print(f'Features: {feature_names}')
print(f'Class balance: {y.mean():.1%} positive')

In [ ]:
# Standardise features — important for consistent CV evaluation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Base estimator: Random Forest (fast, handles non-linearity)
# n_estimators=50 for speed during the search; final eval uses 200
BASE_ESTIMATOR = RandomForestClassifier(
    n_estimators=50,
    max_depth=5,
    random_state=42,
    n_jobs=1,  # Serial inside — we parallelise the outer loop
)

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Base estimator and CV configured.')

## 2. Implement the Three Methods from Scratch

We implement SFS, SBS, and SFFS without using any library's built-in selector. This makes the decision logic fully transparent.

**Key internal data structure:** a Boolean mask array of length $p$, where `mask[j] = True` means feature $j$ is currently selected. An evaluation cache (dict keyed on sorted index tuples) avoids redundant model fits.

In [ ]:
# --- Shared utilities ---

def cv_score(X, y, mask, estimator, cv):
    """
    Cross-validate estimator on the feature subset defined by `mask`.

    Returns the mean CV accuracy.  Returns -inf for empty subsets.
    """
    selected = np.where(mask)[0]
    if len(selected) == 0:
        return -np.inf
    scores = cross_val_score(
        clone(estimator), X[:, selected], y, cv=cv, scoring='accuracy'
    )
    return float(scores.mean())


class EvalCache:
    """Dict cache keyed on sorted feature index tuples."""

    def __init__(self):
        self._store = {}
        self.hits = 0
        self.misses = 0

    def key(self, mask):
        return tuple(np.where(mask)[0])

    def get(self, mask):
        k = self.key(mask)
        if k in self._store:
            self.hits += 1
            return self._store[k]
        self.misses += 1
        return None

    def set(self, mask, score):
        self._store[self.key(mask)] = score

    @property
    def hit_rate(self):
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0.0


def cached_score(X, y, mask, estimator, cv, cache):
    """Return CV score from cache if available, else compute and cache."""
    result = cache.get(mask)
    if result is None:
        result = cv_score(X, y, mask, estimator, cv)
        cache.set(mask, result)
    return result


print('Utilities defined.')

In [ ]:
def sfs(X, y, estimator, cv, k_target):
    """
    Sequential Forward Selection.

    Greedy: at each step, add the feature that maximises CV score.
    Returns (selected_features, score_history, path_history).

    score_history[k] = best CV score with k features
    path_history = list of (step, action, feature_index, score)
    """
    p = X.shape[1]
    mask = np.zeros(p, dtype=bool)
    cache = EvalCache()
    score_history = {}  # size -> score
    path = []           # (step, 'add', feature_idx, score)

    for step in range(k_target):
        candidates = np.where(~mask)[0]
        best_score = -np.inf
        best_j = None

        for j in candidates:
            candidate_mask = mask.copy()
            candidate_mask[j] = True
            score = cached_score(X, y, candidate_mask, estimator, cv, cache)
            if score > best_score:
                best_score = score
                best_j = j

        mask[best_j] = True
        k = int(mask.sum())
        score_history[k] = best_score
        path.append((step + 1, 'add', best_j, best_score))

    return np.where(mask)[0], score_history, path, cache


print('SFS defined.')

In [ ]:
def sbs(X, y, estimator, cv, k_target):
    """
    Sequential Backward Selection.

    Start from all features. At each step, remove the feature whose deletion
    causes the least performance loss (equivalently, removing it still gives
    the highest score).
    """
    p = X.shape[1]
    mask = np.ones(p, dtype=bool)  # Start with all features
    cache = EvalCache()
    score_history = {p: cached_score(X, y, mask, estimator, cv, cache)}
    path = []

    step = 0
    while mask.sum() > k_target:
        selected_idx = np.where(mask)[0]
        best_score = -np.inf
        best_j = None

        for j in selected_idx:
            candidate_mask = mask.copy()
            candidate_mask[j] = False
            score = cached_score(X, y, candidate_mask, estimator, cv, cache)
            if score > best_score:
                best_score = score
                best_j = j

        mask[best_j] = False
        k = int(mask.sum())
        score_history[k] = best_score
        path.append((step + 1, 'remove', best_j, best_score))
        step += 1

    return np.where(mask)[0], score_history, path, cache


print('SBS defined.')

In [ ]:
def sffs(X, y, estimator, cv, k_target):
    """
    Sequential Floating Forward Selection.

    After each forward addition, apply a conditional backward phase:
    remove the feature whose deletion improves the best score seen
    at the resulting (smaller) subset size.  Repeat until no beneficial
    removal exists.

    The floating mechanism allows features to be added AND removed,
    escaping the nesting trap of basic SFS.
    """
    p = X.shape[1]
    mask = np.zeros(p, dtype=bool)
    cache = EvalCache()
    best_at_size = {}  # size -> best score ever seen at that size
    path = []
    step = 0

    while mask.sum() < k_target:
        # --- Forward phase: add the best feature ---
        candidates = np.where(~mask)[0]
        if len(candidates) == 0:
            break

        best_score = -np.inf
        best_j = None
        for j in candidates:
            candidate_mask = mask.copy()
            candidate_mask[j] = True
            score = cached_score(X, y, candidate_mask, estimator, cv, cache)
            if score > best_score:
                best_score = score
                best_j = j

        mask[best_j] = True
        k = int(mask.sum())
        best_at_size[k] = max(best_at_size.get(k, -np.inf), best_score)
        path.append((step, 'add', best_j, best_score))
        step += 1

        # --- Backward phase (floating): remove if beneficial ---
        backward_improved = True
        while backward_improved and mask.sum() > 1:
            backward_improved = False
            selected_idx = np.where(mask)[0]
            best_bwd_score = -np.inf
            best_bwd_j = None

            for j in selected_idx:
                candidate_mask = mask.copy()
                candidate_mask[j] = False
                score = cached_score(X, y, candidate_mask, estimator, cv, cache)
                if score > best_bwd_score:
                    best_bwd_score = score
                    best_bwd_j = j

            k_smaller = int(mask.sum()) - 1
            # Only remove if strictly improves best known score at smaller size
            if best_bwd_score > best_at_size.get(k_smaller, -np.inf):
                mask[best_bwd_j] = False
                best_at_size[k_smaller] = best_bwd_score
                path.append((step, 'remove', best_bwd_j, best_bwd_score))
                step += 1
                backward_improved = True

    # Build final score history from best_at_size
    score_history = best_at_size
    return np.where(mask)[0], score_history, path, cache


print('SFFS defined.')

## 3. Run All Three Methods and Compare

We target $k^* = 5$ features from 8. The dataset is small enough that all three methods run in seconds.

In [ ]:
K_TARGET = 5  # Select 5 features from 8

results = {}

for method_name, method_fn in [('SFS', sfs), ('SBS', sbs), ('SFFS', sffs)]:
    t0 = time.time()
    selected, score_hist, path, cache = method_fn(
        X_scaled, y, BASE_ESTIMATOR, CV, K_TARGET
    )
    elapsed = time.time() - t0

    results[method_name] = {
        'selected': selected,
        'score_history': score_hist,
        'path': path,
        'cache': cache,
        'time': elapsed,
    }

    print(f'{method_name}:')
    print(f'  Selected: {[feature_names[i] for i in selected]}')
    print(f'  Score at k={K_TARGET}: {score_hist.get(K_TARGET, "?"):.4f}')
    print(f'  Time: {elapsed:.2f}s | Cache hit rate: {cache.hit_rate:.1%}')
    print(f'  Search steps: {len(path)}')
    print()

In [ ]:
# Validate on a held-out test set using the final selected features
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

eval_model = RandomForestClassifier(n_estimators=200, random_state=42)

print('Test accuracy with k=5 selected features:')
print('-' * 50)

# Baseline: all 8 features
eval_model.fit(X_train, y_train)
baseline_acc = eval_model.score(X_test, y_test)
print(f'All 8 features (baseline): {baseline_acc:.4f}')

for method_name, res in results.items():
    sel = res['selected']
    eval_model.fit(X_train[:, sel], y_train)
    acc = eval_model.score(X_test[:, sel], y_test)
    selected_names = [feature_names[i] for i in sel]
    print(f'{method_name} ({len(sel)} features): {acc:.4f}  → {selected_names}')

## 4. Visualise the Search Path

The search path shows which features were added (green arrow) and removed (red arrow) at each step. SFFS will show both additions and removals — basic SFS shows only additions, SBS only removals.

In [ ]:
def plot_search_path(path, feature_names, title, ax):
    """
    Visualise the search path as a heatmap-style grid.

    Rows = features.  Columns = search steps.
    Green cell = feature in current set (just added).
    Red cell = feature just removed.
    Grey cell = feature present from previous step.
    White cell = feature not selected.
    """
    p = len(feature_names)
    n_steps = len(path)

    # Build state matrix: 0=absent, 1=present, 2=just added, -1=just removed
    # Track current mask through the path
    state_matrix = np.zeros((p, n_steps), dtype=int)
    current_mask = np.zeros(p, dtype=bool)

    for col, (step_num, action, feature_idx, score) in enumerate(path):
        if action == 'add':
            current_mask[feature_idx] = True
            state_matrix[:, col] = current_mask.astype(int)  # 1=present
            state_matrix[feature_idx, col] = 2               # 2=just added
        elif action == 'remove':
            current_mask[feature_idx] = False
            state_matrix[:, col] = current_mask.astype(int)
            state_matrix[feature_idx, col] = -1              # -1=just removed

    # Custom colormap: white, grey, green, red
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(['#e74c3c', '#ffffff', '#cccccc', '#27ae60'])
    # Values: -1 → red, 0 → white, 1 → grey, 2 → green
    # Shift by 1 for colormap indexing: -1+1=0, 0+1=1, 1+1=2, 2+1=3

    im = ax.imshow(
        state_matrix + 1,
        aspect='auto',
        cmap=cmap,
        vmin=0, vmax=3,
        interpolation='nearest'
    )

    ax.set_yticks(range(p))
    ax.set_yticklabels(feature_names, fontsize=9)
    ax.set_xlabel('Search step', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')

    # Annotate scores above each step
    for col, (_, _, _, score) in enumerate(path):
        ax.text(col, -0.7, f'{score:.3f}', ha='center', va='top', fontsize=7, rotation=90)

    return im


fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (method_name, res) in zip(axes, results.items()):
    plot_search_path(res['path'], feature_names, method_name, ax)

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#27ae60', label='Added at this step'),
    mpatches.Patch(facecolor='#e74c3c', label='Removed at this step (SFFS)'),
    mpatches.Patch(facecolor='#cccccc', label='Present (carried from prev step)'),
    mpatches.Patch(facecolor='white', edgecolor='black', label='Not selected'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=9,
           bbox_to_anchor=(0.5, -0.05))

plt.suptitle('Search Path: SFS, SBS, SFFS', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('\nNote: Red cells appear only in SFFS (backward phase removes features).')
print('SFS and SBS make only monotone decisions (no reversals).')

## 5. Score vs Subset Size

Plot the best CV score achieved at each subset size for all three methods. This shows whether the floating mechanism in SFFS finds better subsets at intermediate sizes.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

colors = {'SFS': '#3498db', 'SBS': '#e67e22', 'SFFS': '#27ae60'}
markers = {'SFS': 'o', 'SBS': 's', 'SFFS': '^'}

for method_name, res in results.items():
    hist = res['score_history']
    sizes = sorted(hist.keys())
    scores = [hist[k] for k in sizes]
    ax.plot(
        sizes, scores,
        color=colors[method_name],
        marker=markers[method_name],
        label=method_name,
        linewidth=2,
        markersize=7,
    )

ax.axvline(x=K_TARGET, color='red', linestyle='--', alpha=0.6, label=f'k*={K_TARGET}')
ax.set_xlabel('Number of features selected', fontsize=12)
ax.set_ylabel('CV Accuracy (5-fold)', fontsize=12)
ax.set_title('Score vs Subset Size: SFS / SBS / SFFS', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xticks(range(1, 9))
ax.set_xticklabels(feature_names[:1] + list(range(2, 9)))  # label x-axis by count
ax.set_xticks(range(1, 9))

plt.tight_layout()
plt.show()

print('If SFFS (green) is above SFS (blue) at any size, the floating mechanism helped.')

## 6. Computational Cost Comparison

Time each method and count the number of unique model evaluations. Cache hits are free; unique evaluations each require a full CV fit.

In [ ]:
print('Computational cost comparison:')
print('=' * 65)
print(f'{"Method":<8} {"Time (s)":<12} {"Steps":<10} {"Unique evals":<15} {"Cache hit rate"}')
print('-' * 65)

for method_name, res in results.items():
    cache = res['cache']
    unique_evals = cache.hits + cache.misses  # Total calls to score function
    unique_model_fits = cache.misses          # Actual CV fits (each = 5-fold)
    print(
        f'{method_name:<8} {res["time"]:<12.2f} {len(res["path"]):<10} '
        f'{unique_model_fits:<15} {cache.hit_rate:.1%}'
    )

print()
print('SFFS has more steps than SFS due to the backward phase.')
print('Cache hits indicate the backward phase re-used forward-phase scores.')

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

method_labels = list(results.keys())

# Panel 1: Wall time
times = [results[m]['time'] for m in method_labels]
axes[0].bar(method_labels, times, color=['#3498db', '#e67e22', '#27ae60'])
axes[0].set_ylabel('Wall time (seconds)')
axes[0].set_title('Execution Time')
axes[0].grid(axis='y', alpha=0.3)

# Panel 2: Unique model evaluations (CV fits)
unique_fits = [results[m]['cache'].misses for m in method_labels]
axes[1].bar(method_labels, unique_fits, color=['#3498db', '#e67e22', '#27ae60'])
axes[1].set_ylabel('Unique CV fits (cache misses)')
axes[1].set_title('Model Evaluations')
axes[1].grid(axis='y', alpha=0.3)

# Panel 3: Cache hit rate
hit_rates = [results[m]['cache'].hit_rate * 100 for m in method_labels]
axes[2].bar(method_labels, hit_rates, color=['#3498db', '#e67e22', '#27ae60'])
axes[2].set_ylabel('Cache hit rate (%)')
axes[2].set_title('Cache Effectiveness')
axes[2].set_ylim([0, 100])
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Computational Cost: SFS vs SBS vs SFFS', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Feature Agreement Analysis

Which features are selected by all three methods? Which are method-specific? High agreement indicates robust relevance; low agreement suggests the feature's value depends on what else is selected.

In [ ]:
from collections import Counter

# Count how many methods selected each feature
all_selected = []
for res in results.values():
    all_selected.extend(res['selected'].tolist())

vote_counts = Counter(all_selected)

fig, ax = plt.subplots(figsize=(9, 4))

bar_colors = [
    '#27ae60' if vote_counts.get(j, 0) == 3 else
    '#f39c12' if vote_counts.get(j, 0) == 2 else
    '#e74c3c'
    for j in range(len(feature_names))
]

ax.bar(
    feature_names,
    [vote_counts.get(j, 0) for j in range(len(feature_names))],
    color=bar_colors,
)
ax.set_ylabel('Number of methods selecting this feature (out of 3)', fontsize=11)
ax.set_title('Feature Agreement Across SFS, SBS, SFFS', fontsize=12)
ax.set_ylim([0, 3.5])
ax.axhline(y=3, color='green', linestyle='--', alpha=0.5, label='All 3 agree')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('\nGreen = selected by all 3 methods (high confidence relevance)')
print('Orange = selected by 2 methods')
print('Red = selected by only 1 method (path-dependent choice)')

## 8. Exercise: Implement Custom Stopping Criterion

**Task:** Modify the `sfs` function above to add a patience-based early stopping criterion. Stop the search when the score has not improved by more than `tol = 0.001` for `patience = 3` consecutive steps.

**Expected output:** A modified `sfs_with_stopping` function that halts before reaching `k_target` if the plateau criterion fires, and prints how many steps were saved.

In [ ]:
# YOUR CODE HERE
# ---------------
def sfs_with_stopping(X, y, estimator, cv, k_target, patience=3, tol=1e-3):
    """
    SFS with patience-based early stopping.

    Stop early if CV score does not improve by more than `tol`
    for `patience` consecutive steps.

    Returns same tuple as sfs().
    """
    # TODO: implement
    pass

In [ ]:
# AUTO-GRADED TEST — do not modify
def test_sfs_with_stopping():
    assert sfs_with_stopping is not None, 'Define sfs_with_stopping'
    result = sfs_with_stopping(X_scaled, y, BASE_ESTIMATOR, CV, k_target=8,
                                patience=3, tol=1e-3)
    assert result is not None, 'Function must return a result'
    selected, score_hist, path, cache = result
    assert selected is not None and len(selected) >= 1, 'Must select at least one feature'
    assert isinstance(path, list), 'path must be a list'
    assert len(path) <= 8, 'Cannot have more steps than features'
    print(f'Test passed! Steps taken: {len(path)} (max possible: 8)')
    if len(path) < 8:
        print(f'  Early stopping fired after {len(path)} steps — saved {8 - len(path)} evaluations.')

test_sfs_with_stopping()

## Summary

### Key Takeaways

1. **SFS, SBS, SFFS all use the same building block:** evaluate a feature subset with cross-validated model scoring.
2. **The floating mechanism (SFFS) breaks the nesting constraint:** features can be added and removed, escaping local optima that trap basic SFS.
3. **Caching is essential in SFFS:** the backward phase re-evaluates subsets already scored during forward steps — cache hit rates of 20–40% are typical.
4. **SFFS overhead is 1.5–3× SFS** in terms of unique model fits — usually worth the improved solution quality.
5. **High inter-method agreement signals robust feature relevance;** features selected by only one method depend on the search path and may not generalise.

### What's Next

Notebook 02 covers Boruta — a statistically rigorous wrapper that asks which features are *relevant at all* (not just which subset is minimal-optimal) — and Optuna-driven wrapper selection, which treats the feature mask as a hyperparameter.

### Additional Resources

- Guide 01: Sequential search algorithms (full theory and code reference)
- [scikit-learn SequentialFeatureSelector](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html) — production SFS/SBS implementation
- Pudil et al. (1994) — original SFFS paper: *Pattern Recognition Letters* 15(11)